In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

BASE = '/content/drive/MyDrive/Phishing_Detection_Project'

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import accuracy_score, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

Mounted at /content/drive


In [2]:
df = pd.read_csv(f"{BASE}/data/processed/engineered_data.csv")

print(df.shape)
df.head()

(11054, 13)


,class,HTTPS,AnchorURL,PrefixSuffix-,WebsiteTraffic,SubDomains,RequestURL,LinksInScriptTags,ServerFormHandler,GoogleIndex,AgeofDomain,PageRank,DomainRegLen
0,-1,1,0,-1,0,0,1,-1,-1,1,-1,-1,-1
1,-1,-1,0,-1,1,-1,1,-1,-1,1,1,-1,-1
2,-1,-1,0,-1,1,-1,-1,0,-1,1,-1,-1,1
3,1,1,0,-1,0,1,1,0,-1,1,-1,-1,-1
4,1,1,0,-1,1,1,1,0,-1,1,1,-1,-1


In [3]:
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

y = y.replace(-1, 0)

print(X.shape, y.shape)
print(y.value_counts())

(11054, 12) (11054,)
DomainRegLen
0    7388
1    3666
Name: count, dtype: int64


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape)

(8843, 12) (2211, 12)


In [5]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

joblib.dump(scaler, f"{BASE}/models/scaler.pkl")

print("Scaler saved")

Scaler saved


In [6]:
lr = LogisticRegression(max_iter=2000)
lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)
lr_acc = accuracy_score(y_test, lr_pred)

print("LR Accuracy:", round(lr_acc*100,2), "%")

LR Accuracy: 80.37 %


In [7]:
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_acc = accuracy_score(y_test, rf_pred)

print("RF Accuracy:", round(rf_acc*100,2), "%")

RF Accuracy: 82.86 %


In [8]:
svm = SVC(kernel='rbf')

svm.fit(X_train, y_train)

svm_pred = svm.predict(X_test)
svm_acc = accuracy_score(y_test, svm_pred)

print("SVM Accuracy:", round(svm_acc*100,2), "%")

SVM Accuracy: 80.78 %


In [9]:
knn = KNeighborsClassifier(n_neighbors=5)

knn.fit(X_train, y_train)

knn_pred = knn.predict(X_test)
knn_acc = accuracy_score(y_test, knn_pred)

print("KNN Accuracy:", round(knn_acc*100,2), "%")

KNN Accuracy: 80.82 %


In [10]:
results = pd.DataFrame({
    'Model': ['Logistic Regression','Random Forest','SVM','KNN'],
    'Accuracy (%)': [
        round(lr_acc*100,2),
        round(rf_acc*100,2),
        round(svm_acc*100,2),
        round(knn_acc*100,2)
    ]
})

print(results)

                 Model  Accuracy (%)
0  Logistic Regression         80.37
1        Random Forest         82.86
2                  SVM         80.78
3                  KNN         80.82


In [11]:
results.to_csv(f"{BASE}/outputs/model_results.csv", index=False)

joblib.dump(rf, f"{BASE}/models/best_model.pkl")

print("Saved model_results.csv")
print("Saved best_model.pkl")

Saved model_results.csv
Saved best_model.pkl


In [12]:
print(classification_report(y_test, rf_pred))

              precision    recall  f1-score   support

           0       0.88      0.86      0.87      1478
           1       0.73      0.77      0.75       733

    accuracy                           0.83      2211
   macro avg       0.81      0.81      0.81      2211
weighted avg       0.83      0.83      0.83      2211

